In [ ]:
import os, copy
import glob

import numpy as np
import scipy.optimize as so
import pandas as pd
import xarray as xr
import rioxarray as rxr

import netCDF4
import h5py
from osgeo import gdal

#%matplotlib widget
import matplotlib as mpl
import matplotlib.pyplot as plt
import colorcet as cc

import hgrs.driver as driver
import hgrs

opj = os.path.join
hgrs.__version__

In [ ]:
auxdir = hgrs.__path__[0]
auxdir

In [ ]:
workdir = '/data/satellite/acix-iii/Garda'
l1c = 'PRS_L1_STD_OFFL_20210721102700_20210721102705_0001.he5'
l2c = 'PRS_L2C_STD_20210721102700_20210721102705_0001.he5'

In [ ]:
l1c_path = opj(workdir,l1c)
l2c_path = opj(workdir,l2c)

dc_l1c = driver.read_L1C_data(l1c_path,reflectance_unit=True)
dc_l2c = driver.read_L2C_data(l2c_path)
dc_l2c

In [ ]:
coarsening=1
gamma=0.2
brightness_factor = 1
fig = (dc_l1c.Rtoa[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))
fig.axes.set(xticks=[], yticks=[])
fig.axes.set_ylabel('')
fig.axes.set_xlabel('')
fig

In [ ]:
prisma_rsr = dc_l1c.fwhm.to_dataframe()

In [ ]:
def Gamma2sigma(Gamma):
    '''Function to convert FWHM (Gamma) to standard deviation (sigma)'''
    return Gamma * np.sqrt(2.) / ( np.sqrt(2. * np.log(2.)) * 2. )

def gaussian(x,mu,sigma):
    return 1 / (sigma * np.sqrt(2*np.pi)) * np.exp(-(x-mu)**2/(2*sigma**2))

wl_ref = np.linspace(360,2550,10000)
fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(10, 4))
rho_int=[]
for mu,fwhm in prisma_rsr.iterrows():
    sig = Gamma2sigma(fwhm.values)
    rsr = gaussian(wl_ref,mu,sig)
    #rho_ = np.trapz(rho * rsr, wl_ref)/np.trapz(rsr, wl_ref)
    #rho_int.append(rho_)
    axs.plot(wl_ref,rsr,'-k',lw=0.5,alpha=0.4)

In [ ]:
%matplotlib widget
plt.figure()
dc_l1c.F0.plot()
F0_k = driver.SolarIrradiance().kurucz
F0_k.plot(lw=0.5)
wl_ref = F0_k.wl.values
F0_int=[]
for mu,fwhm in prisma_rsr.iterrows():
    sig = Gamma2sigma(fwhm.values)
    rsr = gaussian(wl_ref,mu,sig)

    F0_ = (F0_k * rsr).integrate('wl')/np.trapz(rsr, wl_ref)
    F0_int.append(F0_.values)
plt.plot(dc_l1c.wl,F0_int)

In [ ]:
abs_file = '/DATA/git/vrtc/libradtran_tbx/output/lut_abs_opt_thickness_normalized.nc' 
ot =xr.open_dataset(abs_file)
ot

In [ ]:
sza=40
vza=14
tcwv=20
to3c=6.5e-3
tno2c=3e-6
tch4c= 1e-2
pressure=1000

In [ ]:
M=1./np.cos(np.radians(sza))+1./np.cos(np.radians(vza))
variable='o4'
variables=['ch4','co','co2','h2o','n2o','no2','o2','o3','o4']
conc={'ch4':tch4c,'co':1,'co2':1,
      'h2o':tcwv,'n2o':1,'no2':tno2c,
      'o2':1,'o3':to3c,'o4':1}

fig,axs = plt.subplots(3,3,figsize=(30,15),sharex=True)
axs=axs.ravel()

for i, variable in enumerate(variables):
    abs_ot = ot[variable]*conc[variable]


    Ttot = (np.exp(-M*abs_ot)) 
    Ttot.plot(ax=axs[i],color='grey',lw=0.5)

    axs[i].minorticks_on()
    axs[i].set_title(variable)
plt.legend()

## Convert OT for actual gas content (in kg/m2) and pressure (in hPa)

In [ ]:
def get_pressure(alt, psl):
    '''Compute the pressure for a given altitude
       alt : altitude in meters (float or np.array)
       psl : pressure at sea level in hPa
       palt : pressure at the given altitude in hPa'''

    palt = psl * (1. - 0.0065 * np.nan_to_num(alt) / 288.15) ** 5.255
    return palt
psl=1010
alt=60
get_pressure(alt, psl)

In [ ]:


#['ch4','co','co2','h2o','n2o','no2','o2','o3','o4']

ot_wv = ot.h2o *tcwv
ot_o3 = ot.o3 * to3c
ot_ch4 = ot.ch4 * tch4c
ot_no2 = ot.no2 * tno2c
ot_others = (ot.co+ot.co2+ot.n2o+ot.o2+ot.o4)* pressure/1000

ot_tot = ot_wv+ot_ch4+ot_no2+ot_o3+ot_others 

M=2
fig,axs = plt.subplots(nrows=2,figsize=(15,10))
Ttot = np.exp(-M*ot_tot)
Ttot.plot(ax=axs[0],color='grey',lw=0.5)
Ttot.plot(ax=axs[1],color='grey',lw=0.5)
axs[1].set_xlim(350,1000)

## Convolution of the transmittance with sensor spectral response

In [ ]:
rsr

In [ ]:
sza=27.7
vza=14
M=1./np.cos(np.radians(sza))+1./np.cos(np.radians(vza))
Ttot = np.exp(-M*ot_tot)
wl_ref = ot_tot.wl#.values
Ttot_int=[]  
for mu,fwhm in prisma_rsr.iterrows():
    sig = Gamma2sigma(fwhm.values)
    rsr = gaussian(wl_ref,mu,sig)
    
    Ttot_ = (Ttot * rsr).integrate('wl')/np.trapz(rsr, wl_ref)
    Ttot_int.append(Ttot_.values)
Ttot_sat = xr.DataArray(Ttot_int,name='Ttot',coords={'wl':dc_l1c.wl.values})

In [ ]:

fig,axs = plt.subplots(nrows=2,figsize=(15,10))

Ttot_sat.plot(ax=axs[0],color='grey',lw=1.5)
Ttot_sat.plot(ax=axs[1],color='grey',lw=1.5)
axs[1].set_xlim(350,1000)

In [ ]:
sza=50
vza=0
M=1./np.cos(np.radians(sza))+1./np.cos(np.radians(vza))
variables=['ch4','co','co2','h2o','n2o','no2','o2','o3','o4']
conc={'ch4':tch4c,'co':1,'co2':1,
      'h2o':tcwv,'n2o':1,'no2':tno2c,
      'o2':1,'o3':to3c,'o4':1}

fig,axs = plt.subplots(3,3,figsize=(30,15),sharex=True)
axs=axs.ravel()

for i, variable in enumerate(variables):
    abs_ot = ot[variable]*conc[variable]

    axs[i].grid(visible=True)
    Ttot = (np.exp(-M*abs_ot)) 
    Ttot_int=[]  
    for mu,fwhm in prisma_rsr.iterrows():
        sig = Gamma2sigma(fwhm.values)
        rsr = gaussian(wl_ref,mu,sig)

        Ttot_ = (Ttot * rsr).integrate('wl')/np.trapz(rsr, wl_ref)
        Ttot_int.append(Ttot_.values)
    Ttot_sat = xr.DataArray(Ttot_int,name='Ttot',coords={'wl':dc_l1c.wl.values})
    Ttot_sat.plot(ax=axs[i],color='red',lw=1.5)

    axs[i].minorticks_on()
    axs[i].set_title(variable)

#fig.savefig('/DATA/projet/magellium/obs2co/roadmap/abs_gas/transmittance_retran_prisma.png',dpi=300)

In [ ]:
sza=27.7
vza=14
M=1./np.cos(np.radians(sza))+1./np.cos(np.radians(vza))
Ttot = np.exp(-M*ot_tot)
wl_ref = ot_tot.wl#.values
Twv=[]
for tcwv in [0,1,5,10,20,40,60]:
    ot_wv = ot.h2o *tcwv
    Ttot = np.exp(-M*ot_wv)
    Ttot_int=[]  
    for mu,fwhm in prisma_rsr.iterrows():
        sig = Gamma2sigma(fwhm.values)
        rsr = gaussian(wl_ref,mu,sig)

        Ttot_ = (Ttot * rsr).integrate('wl')/np.trapz(rsr, wl_ref)
        Ttot_int.append(Ttot_.values)
        
    Twv.append(xr.DataArray(Ttot_int,name='Twv',coords={'wl':dc_l1c.wl.values}).assign_coords({'tcwv':tcwv}))    
Twv = xr.concat(Twv,dim='tcwv')

In [ ]:
plt.ioff()

cmap=plt.cm.Spectral_r
norm = mpl.colors.Normalize(vmin=0.1, vmax=60)
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

fig,axs = plt.subplots(nrows=3,figsize=(15,15))
for tcwv in [0.1,1,5,10,20,40,60]:
   
    Ttot_sat = Twv.interp(tcwv=tcwv) #Ttot_int,name='Ttot',coords={'wl':dc_l1c.wl.values})
    Ttot_sat.plot(ax=axs[0],color=cmap(norm(tcwv)),label='tcwv={:.1f}'.format(tcwv))
    Ttot_sat.plot(ax=axs[1],color=cmap(norm(tcwv)))
    Ttot_sat.plot(ax=axs[2],color=cmap(norm(tcwv)))
axs[1].set_xlim(350,1000)
axs[2].set_xlim(850,1000)

for i in range(3):
    axs[i].minorticks_on()
    axs[i].set_xlabel('Wavelength (nm)')
    axs[i].set_title('')
fig.colorbar(sm,shrink=0.5,label='$Total\ water\ vapor\ (kg\cdot m^{-2})$' , anchor=(0,0),location='top')   
plt.show()

In [ ]:
from scipy.optimize import curve_fit

def func(tcwv,a,b):
    return np.exp(a*tcwv**b)

Twv_ = Twv.sel(wl=slice(850,970))
plt.figure()
for wl in Twv_.wl.values:
    data = Twv.sel(wl=wl,method='nearest')

    popt, pcov = curve_fit(func, data.tcwv, data.values)
    popt
   

    data.plot(marker='o',hue='wl')
    tcwvs=np.linspace(0,60,100)
    plt.plot(tcwvs,func(tcwvs,*popt),'k--')

In [ ]:
Twv_ = Twv.sel(wl=slice(800,1300))
wl_ = Twv_.wl.values
def toa_simu(wl_,tcwv,a,b):
    
    return Twv_.interp(tcwv=tcwv).values *(  (a*(wl_/1000)+b))
    
tcwv=20
a=-0.04
b=0.093

c=0.04
plt.figure()   
pixel = dc_l1c.isel(x=600,y=600)
pixel.Rtoa.plot(lw=1,color='black')
plt.plot(wl_,toa_simu(wl_,tcwv,a,b),'r--',lw=1)  


In [ ]:
from scipy.optimize import least_squares


def toa_simu(wl,tcwv,a,b):
    Twv_ = Twv.sel(wl=wl)
    return Twv_.interp(tcwv=tcwv).values * (a*(wl/1000)+b)

def fun(x, wl, y):
    return toa_simu(wl,*x) - y

plt.figure() 

img = dc_l1c.Rtoa.sel(wl=slice(800,1200))
wl_= img.wl.values

for y in [50,100,150,200,250,350,400,450,500,550]:
    data = img.isel(x=600,y=y)
    
    
    x0 = np.array([20,-0.04,0.1])
    res_lsq = least_squares(fun, x0, args=(data.wl.values, data.values),diff_step=1e-3,xtol=1e-2,ftol=1e-2)

  
    data.plot(lw=1,color='black')
    plt.plot(wl_,toa_simu(wl_,*res_lsq.x),'r--',lw=1)  
    print(res_lsq.x,res_lsq.nfev)


In [ ]:
img

In [ ]:
from scipy.optimize import least_squares

def toa_simu(wl,tcwv,a,b):
    Twv_ = Twv.sel(wl=wl)

    return Twv_.interp(tcwv=tcwv).values * (a*(wl/1000)+b)

def fun(x, wl, y):
    return toa_simu(wl,*x) - y

plt.figure() 

img = dc_l1c.Rtoa.sel(wl=slice(800,1200)).coarsen(x=25, y=25).mean()
wl_= img.wl.values
data =img.values
width,height,nwl = data.shape
res = np.full((width,height),np.nan)
x0 = np.array([20,-0.04,0.1])
for iy in range(height):
    print(iy)
    for ix in range(width):
        #print(data[ix,iy])
        res_lsq = least_squares(fun, x0, args=(wl_, data[ix,iy]),diff_step=1e-3,xtol=1e-2,ftol=1e-2,max_nfev=5)
        x0 = res_lsq.x
        res[ix,iy] = x0[0]
        


In [ ]:
x0

In [ ]:
fig,axs = plt.subplots(1,2,figsize=(15,6))
im= axs[0].imshow(res[:,::-1],cmap=plt.cm.YlGnBu)
p = (dc_l1c.Rtoa[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True,ax=axs[1])#, subplot_kws=dict(projection= l1c.proj))
fig.colorbar(im, ax=axs[0], label='$tcwv\ (kg\cdot m^{-2})$')

In [ ]:
(img/img.sel(wl=902,method='nearest')).plot(col='wl',col_wrap=7,cmap=plt.cm.RdBu)


In [ ]:
data = pixel.Rtoa.sel(wl=slice(800,1300))
popt, pcov = curve_fit(toa_simu, data.wl, data.values,p0=[20,-0.04,.09])
print(popt)
plt.figure()   
data.plot(lw=1,color='black')
plt.plot(wl_,toa_simu(wl_,*popt),'r--',lw=1)  

In [ ]:
plt.figure()
ratio = Twv.sel(wl=slice(850,970))#/Twv.sel(wl=940,method='nearest')




ratio.plot(marker='o',hue='wl')
tcwvs=np.linspace(0,60,100)
plt.plot(tcwvs,func(tcwvs,[-0.02,0.65]),'k--')

In [ ]:
tcwvs=np.linspace(0,60,100)
plt.plot(tcwvs,1-0.021*tcwvs+0.0005*tcwvs**2)

 ## Application to satellite data

In [ ]:
%matplotlib widget

In [ ]:
pixel = dc_l1c.isel(x=600,y=600)
dc_l2c.isel(x=600,y=600)


In [ ]:


#['ch4','co','co2','h2o','n2o','no2','o2','o3','o4']

ot_wv = ot.h2o *tcwv
ot_o3 = ot.o3 * to3c
ot_ch4 = ot.ch4 * tch4c
ot_no2 = ot.no2 * tno2c
ot_others = (ot.co+0.5*ot.co2+ot.n2o+0.5*ot.o2+ot.o4)* pressure/1000

ot_tot = ot_ch4+ot_no2+ot_o3+ot_others 

sza=27.6
vza=13.84
M=1./np.cos(np.radians(sza))+1./np.cos(np.radians(vza))
Tg = np.exp(-M*ot_tot)

Tg_int=[]
for mu,fwhm in prisma_rsr.iterrows():
    sig = Gamma2sigma(fwhm.values)
    rsr = gaussian(wl_ref,mu,sig)

    Tg_ = (Tg * rsr).integrate('wl')/np.trapz(rsr, wl_ref)
    Tg_int.append(Tg_.values)
Tg_sat = xr.DataArray(Tg_int,name='Ttot',coords={'wl':dc_l1c.wl.values})


In [ ]:
Twv=[]
for tcwv in [0,5,10,20,40,60]:
    ot_wv = ot.h2o *tcwv
    Ttot = np.exp(-M*ot_wv)
    Twv_int=[]
    for mu,fwhm in prisma_rsr.iterrows():
        sig = Gamma2sigma(fwhm.values)
        rsr = gaussian(wl_ref,mu,sig)

        Ttot_ = (Ttot * rsr).integrate('wl')/np.trapz(rsr, wl_ref)
        Twv_int.append(Ttot_.values)
    Twv.append(xr.DataArray(Twv_int,name='Twv',coords={'wl':dc_l1c.wl.values}).assign_coords({'tcwv':tcwv}))

In [ ]:
Twv = xr.concat(Twv,dim='tcwv')
Twv =Twv.where(Twv > 0.05)

In [ ]:

lw=0.5
fig,axs = plt.subplots(nrows=1,figsize=(15,5))
axs.minorticks_on()
pixel.Rtoa.plot(lw=lw,color='black',ax=axs)

(pixel.Rtoa/Tg_sat).plot(lw=lw,color='grey',ax=axs)
(pixel.Rtoa/Tg_sat/Twv.interp(tcwv=22)).plot(lw=lw,color='blue',ax=axs)
(pixel.Rtoa/Tg_sat/Twv.interp(tcwv=18)).plot(lw=lw,color='green',ax=axs)
(pixel.Rtoa/Tg_sat/Twv.interp(tcwv=20)).plot(lw=lw,color='red',ax=axs)

In [ ]:
plt.figure()
ratio = Twv.sel(wl=870,method='nearest')/Twv.sel(wl=940,method='nearest')
ratio_sat = pixel.Rtoa.sel(wl=870,method='nearest')/pixel.Rtoa.sel(wl=940,method='nearest')
ratio.plot(marker='o')

In [ ]:
ratio_sat

In [ ]:
Ttot_sat = Tg_sat*Twv.interp(tcwv=20)
img = dc_l1c[['Rtoa','Ltoa']] / Ttot_sat 

In [ ]:
img = img.where(Ttot_sat > 0.05)
img

In [ ]:
ratio_sat = dc_l1c.Rtoa.sel(wl=940,method='nearest')/dc_l1c.Rtoa.sel(wl=870,method='nearest')#-)/(dc_l1c.Rtoa.sel(wl=870,method='nearest')+dc_l1c.Rtoa.sel(wl=940,method='nearest'))
plt.figure()
ratio_sat.plot(robust=True)

In [ ]:
from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
from holoviews import opts

opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))
# set the parameter for spectra extraction
hv.extension('bokeh')
pn.extension()

param = 'Rtoa'
raster = dc_l1c[param] 
#raster = img[param]  
vmax=0.3
#param = 'rho'
#raster = dc_l2c[param] 

third_dim = 'wl'

wl= raster.wl.data
Nwl = len(wl)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= 'RdBu_r',colorbar=True,clim=(0,vmax)).hist(bin_range=(0,vmax/4)) 

polys = hv.Polygons([])
box_stream = hv.streams.BoxEdit(source=polys)
dmap, dmap_std=[],[]

def roi_curves(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]= hv.Curve((wl,mean[param]),'Wavelength (nm)', param) 

    return hv.NdOverlay(curves)


# a bit dirty to have two similar function, but holoviews does not like mixing Curve and Spread for the same stream
def roi_spreads(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]=  hv.Spread((wl,mean[param],std[param]),fill_alpha=0.3)

    return hv.NdOverlay(curves)

mean=hv.DynamicMap(roi_curves,streams=[box_stream])
std =hv.DynamicMap(roi_spreads, streams=[box_stream])    
hlines = hv.HoloMap({wl[i]: hv.VLine(wl[i]) for i in range(Nwl)},third_dim )


hv.output(widget_location='top_left')

# visualize and play
graphs = ((mean* std *hlines).relabel(param))
layout = (im * polys +graphs    ).opts(
    opts.Curve(width=800,height=550, framewise=True,xlim=(400,2500)), 
    opts.Polygons(fill_alpha=0.2, color='green',line_color='black'), 
    opts.VLine(color='black')).cols(2)
layout 

## Example of exploiation: compute NDWI for water pixel masking


In [ ]:
# Compute NDWI
green = img.isel(wl=20)
nir = img.isel(wl=55)
ndwi = (green - nir) / (green + nir)

In [ ]:
coarsening=1

# binary cmap
bcmap = mpl.colors.ListedColormap(['khaki', 'lightblue'])

def water_mask(ndwi, threshold=0):
    water = xr.where(ndwi > threshold, 1, 0)
    return water.where(~np.isnan(ndwi))

def plot_water_mask(ndwi,ax,threshold=0):
    water = water_mask(ndwi, threshold)
    #ax.set_extent(extent_val, proj)
    water.plot.imshow( cmap=bcmap,
                                  cbar_kwargs={'ticks': [0, 1], 'shrink': shrink})#extent=extent_val, transform=proj,
    ax.set_title(str(threshold)+' < NDWI')
    
fig = plt.figure(figsize=(20, 15))
fig.subplots_adjust(bottom=0.1, top=0.95, left=0.1, right=0.99,
                    hspace=0.05, wspace=0.05)
shrink = 0.8
    
ax = plt.subplot(2, 2, 1)#, projection=proj)
#ax.set_extent(extent_val, proj)
fig = ndwi[::coarsening, ::coarsening].plot.imshow(cmap=plt.cm.BrBG, robust=True,
                                   cbar_kwargs={'shrink': shrink})# extent=extent_val, transform=proj, 
# axes.coastlines(resolution='10m',linewidth=1)
ax.set_title('Sentinel 2, NDWI')

for i,threshold in enumerate([-0.4,-0.2,0.]):
    ax = plt.subplot(2, 2, i+2)#, projection=proj)
    plot_water_mask(ndwi[::coarsening, ::coarsening],ax,threshold=threshold)

plt.show()



## Plot the top-of-atmosphere (TOA) radiance in mW/m2/nm

In [ ]:
threshold=-0.2
masked = img.where(ndwi > threshold)

In [ ]:
masked
coarsening=1
gamma=0.2
brightness_factor = 1
fig = (masked[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))
fig.axes.set(xticks=[], yticks=[])
fig.axes.set_ylabel('')
fig.axes.set_xlabel('')
fig
plt.show()

In [ ]:

fig = masked.Ltoa.isel(wl=[1,10,20,30,40,50,90,130,200]).plot.imshow(col='wl',col_wrap=3,robust=True,cmap=cc.cm.bky)
for ax in fig.axs.flat:
    ax.set(xticks=[], yticks=[])
    ax.set_ylabel('')
    ax.set_xlabel('')
fig

## Plot the top-of-atmosphere (TOA) reflectance

In [ ]:
fig = masked.Rtoa.isel(wl=[1,10,20,30,40,50,90,130,200]).plot.imshow(col='wl',col_wrap=3,robust=True,cmap=cc.cm.bky)
for ax in fig.axs.flat:
    ax.set(xticks=[], yticks=[])
    ax.set_ylabel('')
    ax.set_xlabel('')
fig

In [ ]:
from hgrs import metadata
auxdata = metadata()#wl=masked.wl)
wlref=2200
wl = masked.wl
sunglint_eps = auxdata.sunglint_eps.interp(wl=wl)
rot = auxdata.rot.interp(wl=wl)

In [ ]:
auxdata.solar_irr.thuillier

In [ ]:

sunglintBRDF =  sunglint_eps['mean'] / sunglint_eps['mean'].sel(wl=wlref,method='nearest')
sunglintBRDF.plot()


In [ ]:
aot550=0.1
ang_exp = 1.1

def aot_angstrom(wl, aotref, ang_exp,wlref=550):
    '''function for spectral variation of AOT'''
    
    return (wl/wlref)**-ang_exp * aotref

aot=aot_angstrom(wl,aot550,ang_exp)

fig, axs = plt.subplots()
aot.plot(ax=axs,label='Aerosol')
rot.plot(ls=':',label='Rayleigh',ax=axs)
axs.set_xlabel('Wavelength (nm)')
axs.set_ylabel('Optical thickness')
plt.legend()

In [ ]:
aot

In [ ]:
def transmittance_dir(aot,sza=30,vza=10,rot=0):
    air_mass = 1/np.cos(np.radians(sza)) + 1/np.cos(np.radians(vza))
    return np.exp(-(rot+aot)*air_mass)
                
Tdir = transmittance_dir(aot,rot=rot)
Tdir.plot()

In [ ]:
sunglint_corr = Tdir * sunglintBRDF
Rcorr = masked.Rtoa - sunglint_corr * masked.Rtoa.sel(wl=wlref,method='nearest')

In [ ]:
fig = Rcorr.isel(wl=[1,10,20,30,40,50,90,130,200]).plot.imshow(col='wl',col_wrap=3,vmin=0,robust=True,cmap=cc.cm.bky)
for ax in fig.axs.flat:
    ax.set(xticks=[], yticks=[])
    ax.set_ylabel('')
    ax.set_xlabel('')
fig

In [ ]:
coarsening=1
brightness_factor = 7.5
(masked.Rtoa[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])*brightness_factor).plot.imshow(rgb='wl')#, subplot_kws=dict(projection= l1c.proj))

In [ ]:
#(Rcorr[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])*brightness_factor).plot.imshow(rgb='wl')#, subplot_kws=dict(projection= l1c.proj))
coarsening=1

gamma=0.2
brightness_factor = 1
fig = (dc_l1c.Rtoa[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])**gamma*brightness_factor).plot.imshow(rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))
(Rcorr[:, ::coarsening, ::coarsening].isel(wl=[30,20,10])).plot.imshow(ax=fig.axes,rgb='wl',robust=True)#, subplot_kws=dict(projection= l1c.proj))
fig.axes.set(xticks=[], yticks=[])
fig.axes.set_ylabel('')
fig.axes.set_xlabel('')
fig

In [ ]:
Rcorr.name='Rtoa'


In [ ]:
from holoviews import streams
import holoviews as hv
import panel as pn
import param
import numpy as np
import xarray as xr
hv.extension('bokeh')
from holoviews import opts

opts.defaults(
    opts.GridSpace(shared_xaxis=True, shared_yaxis=True),
    opts.Image(cmap='binary_r', width=800, height=700),
    opts.Labels(text_color='white', text_font_size='8pt', text_align='left', text_baseline='bottom'),
    opts.Path(color='white'),
    opts.Spread(width=900),
    opts.Overlay(show_legend=True))

raster=Rcorr
param='Rtoa'
third_dim = 'wl'
cmap = cc.cm.CET_L16

wl= raster.wl.data
Nwl = len(wl)
ds = hv.Dataset(raster.persist())
im= ds.to(hv.Image, ['x', 'y'], dynamic=True).opts(cmap= cmap,colorbar=True,clim=(0,0.08)).hist(bin_range=(0,0.02)) 

polys = hv.Polygons([])
box_stream = hv.streams.BoxEdit(source=polys)
dmap, dmap_std=[],[]

def roi_curves(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]= hv.Curve((wl,mean[param]),'Wavelength (nm)', param) 

    return hv.NdOverlay(curves)


# a bit dirty to have two similar function, but holoviews does not like mixing Curve and Spread for the same stream
def roi_spreads(data,ds=ds):    
    if not data or not any(len(d) for d in data.values()):
        return hv.NdOverlay({0: hv.Curve([],'Wavelength (nm)', param)})

    curves,envelope = {},{}
    data = zip(data['x0'], data['x1'], data['y0'], data['y1'])
    for i, (x0, x1, y0, y1) in enumerate(data):
        selection = ds.select(x=(x0, x1), y=(y0, y1))
        mean = selection.aggregate(third_dim, np.mean).data
        std = selection.aggregate(third_dim, np.std).data
        wl = mean.wl

        curves[i]=  hv.Spread((wl,mean[param],std[param]),fill_alpha=0.3)

    return hv.NdOverlay(curves)

mean=hv.DynamicMap(roi_curves,streams=[box_stream])
std =hv.DynamicMap(roi_spreads, streams=[box_stream])    
hlines = hv.HoloMap({wl[i]: hv.VLine(wl[i]) for i in range(Nwl)},third_dim )


hv.output(widget_location='top_left')

# visualize and play
graphs = ((mean* std *hlines).relabel(param))
layout = (im * polys +graphs    ).opts(
    opts.Curve(width=600, framewise=True,xlim=(400,2500),tools=['hover']), 
    opts.Polygons(fill_alpha=0.2, color='green',line_color='black'), 
    opts.VLine(color='black')).cols(2)
layout 